In [1]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
#from textblob import TextBlob, Word
from nltk.stem.snowball import SnowballStemmer

%matplotlib inline

### What Is Natural Language Processing (NLP)?

- Using computers to process (analyze, understand, generate) natural human languages.
- Making sense of human knowledge stored as unstructured text.

### What Are Some of the Higher-Level Task Areas?

Some higher-level tasks include:

- **Question answering:** Determine the intent of the question, match query with knowledge base, evaluate hypotheses.
- **Information retrieval:** Find relevant results and similar results.
- **Machine translation:** One language to another.
- **Predictive text input:** Faster or easier typing.

### Higher-Level Tasks Are Hard Because

- **Ambiguity**:
    - Hospitals Are Sued by 7 Foot Doctors
    - Juvenile Court to Try Shooting Defendant
    - Local High School Dropouts Cut in Half
- **Idioms:** "throw in the towel"
- **Newly coined words & non-standard words:** "retweet"
- **Tricky entity names:** "Where is A Bug's Life playing?"

### Our Use Case: Text Classification
** the task of predicting which category or topic a text sample is from **

In [3]:
# Read yelp.csv into a DataFrame.
path = r'./data/yelp.csv'
yelp = pd.read_csv(path)
yelp.head(1)

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0


In [4]:
#sample 'document'
yelp.text.values[0]

'My wife took me here on my birthday for breakfast and it was excellent.  The weather was perfect which made sitting outside overlooking their grounds an absolute pleasure.  Our waitress was excellent and our food arrived quickly on the semi-busy Saturday morning.  It looked like the place fills up pretty quickly so the earlier you get here the better.\n\nDo yourself a favor and get their Bloody Mary.  It was phenomenal and simply the best I\'ve ever had.  I\'m pretty sure they only use ingredients from their garden and blend them fresh when you order it.  It was amazing.\n\nWhile EVERYTHING on the menu looks excellent, I had the white truffle scrambled eggs vegetable skillet and it was tasty and delicious.  It came with 2 pieces of their griddled bread with was amazing and it absolutely made the meal complete.  It was the best "toast" I\'ve ever had.\n\nAnyway, I can\'t wait to go back!'

First, we vectorize the text into a set of numeric features. 

- For a given document, the numeric value of each feature could be the number of times the word appears in the document.
    - So, most features ['tokens'] will have a value of zero, resulting in a sparse matrix of features.

- This technique for vectorizing text is referred to as a bag-of-words model. 
    - It is called bag of words because the document's structure is lost — as if the words are all jumbled up in a bag.
    
Now we can apply a machine learning classifier.

![DTM](images/DTM.png)

## Preprocessing 
### Relevant to text classification and higher-level tasks alike


- **Tokenization:** Breaking text into tokens (words, sentences, n-grams)

### N-grams

N-grams are features which consist of N consecutive words. This is useful because using the bag-of-words model, treating `data scientist` as a single feature has more meaning than having two independent features `data` and `scientist`!

Example:
```
my cat is awesome
Unigrams (1-grams): 'my', 'cat', 'is', 'awesome'
Bigrams (2-grams): 'my cat', 'cat is', 'is awesome'
Trigrams (3-grams): 'my cat is', 'cat is awesome'
4-grams: 'my cat is awesome'
```
- You set the lower and upper boundary of the range of n-values for different n-grams to be extracted. All values of n such that min_n <= n <= max_n will be used.

### Stop-Word Removal

Stop words are some of the most common words in a language. They are used so that a sentence makes sense grammatically, such as prepositions and determiners, e.g., "to," "the," "and." However, they are so commonly used that they are generally worthless for predicting the class of a document. 

### Capitalization

- Does upper or lower case matter? to a computer, aPPle is a different 'token' than apple

### Setting a cut-off for occurrences
- Extrapolating from infrequently occurring terms can lead to overfitting. When building the vocabulary, ignore terms that have a document frequency strictly lower than the given threshold. 


### Stemming and Lemmatization

Stemming is a crude process of removing common endings from sentences, such as "s", "es", "ly", "ing", and "ed".

- **What:** Reduce a word to its base/stem/root form.
- **Why:** This intelligently reduces the number of features by grouping together (hopefully) related words.

Lemmatization is a more refined process that uses specific language and grammar rules to derive the root of a word.  

This is useful for words that do not share an obvious root such as "better" and "best".

- **What:** Lemmatization derives the canonical form ("lemma") of a word.

**More Lemmatization and Stemming Examples**

|Lemmatization|Stemming|
|-------------|---------|
|shouted → shout|badly → bad|
|best → good|computing → comput|
|better → good|computed → comput|
|good → good|wipes → wip|
|wiping → wipe|wiped → wip|
|hidden → hide|wiping → wip|

In [41]:
# Create a new DataFrame that only contains the 5-star and 1-star reviews.
yelp_best_worst = yelp[(yelp.stars==5) | (yelp.stars==1)]

# Define X and y.
X = yelp_best_worst.text
y = yelp_best_worst.stars

# Split the new DataFrame into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y)

# Use CountVectorizer to create document-term matrices from X_train and X_test. Note parameters.
vect = CountVectorizer(lowercase=True, max_features=None,min_df=2,ngram_range=(1,2),stop_words='english')

# fit: Learn a vocabulary dictionary of all tokens in the raw documents.
# transform: Transform documents to document-term matrix.

# for train we fit *and* transform
X_train_dtm = vect.fit_transform(X_train)
# for test, we transform; we only use vocabulary present in train
X_test_dtm = vect.transform(X_test)
X_train_dtm.shape

(3064, 21641)

In [25]:
# Last 50 tokens in our vocabulary
print((vect.get_feature_names()[-50:]))

['young woman', 'younger', 'youngest', 'yr', 'yr old', 'yrs', 'yuck', 'yucky', 'yuk', 'yukon', 'yukon gold', 'yum', 'yum dinner', 'yum sausage', 'yum yum', 'yumm', 'yummm', 'yummmm', 'yummy', 'yummy bread', 'yummy chicken', 'yummy dessert', 'yummy food', 'yummy happy', 'yummy little', 'yummy meal', 'yummy pizza', 'yummy yummy', 'yung', 'yup', 'zach', 'zen', 'zero', 'zero attitude', 'zero stars', 'zin', 'zinburger', 'zinc', 'zinfandel', 'zip', 'zipps', 'zoe', 'zoe kitchen', 'zone', 'zones', 'zoo', 'zucchini', 'zuchinni', 'zumba', 'zuzu']


In [15]:
# our transformed training data is a big pivot table stored in a memory efficient format. We can convert it to an 
# array or df but its memory allocation would be very expensive
type(X_train_dtm)

scipy.sparse.csr.csr_matrix

In [42]:
# Use Logistic Regression to predict the star rating.
logr = LogisticRegression()
logr.fit(X_train_dtm, y_train)
y_pred_class = logr.predict(X_test_dtm)
print('baseline: '+str(y_test.value_counts('mean').values[0]))
# Calculate accuracy.
print('model accuracy: '+str((accuracy_score(y_test, y_pred_class))))

baseline: 0.8189823874755382
model accuracy: 0.9393346379647749


/anaconda/lib/python3.6/site-packages/sklearn/linear_model/logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


In [43]:
features = np.array(vect.get_feature_names())
logr_coefs = pd.DataFrame({'coef':logr.coef_[0]},index=features)
logr_coefs = logr_coefs.sort_values('coef',ascending=False)
# words that predict positive reviews
logr_coefs.head(10)

,coef
great,1.901709
best,1.747406
awesome,1.577783
amazing,1.468703
excellent,1.425375
love,1.390827
favorite,1.140431
friendly,1.126260
delicious,1.016270
fantastic,0.967172


In [44]:
# words that predict negative reviews
logr_coefs.tail(10)

,coef
gay,-0.852834
bad,-0.864372
wasn,-0.876669
avoid,-0.891629
told,-0.900295
worst,-0.949935
gross,-1.042799
rude,-1.172595
awful,-1.417139
horrible,-1.421287


## Term Frequency–Inverse Document Frequency (TF–IDF)

Term frequency–inverse document frequency (TF–IDF) computes the "relative frequency" with which a word appears in a document, compared to its frequency across all documents.

It's more useful than simple "term frequency" [the count vectorizer above] for identifying "important" words in each document (high frequency in that document, low frequency in other documents). (But not necessarily more performant.)

### Example 

In [27]:
simple_train = ['call you tonight', 'Call me a cab', 'please call me... PLEASE!']

# Term frequency
vect = CountVectorizer()
tf = pd.DataFrame(vect.fit_transform(simple_train).toarray(), columns=vect.get_feature_names())
tf

,cab,call,me,please,tonight,you
0,0,1,0,0,1,1
1,1,1,1,0,0,0
2,0,1,1,2,0,0


In [28]:
# Document frequency
vect = CountVectorizer(binary=True)
df = vect.fit_transform(simple_train).toarray().sum(axis=0)
pd.DataFrame(df.reshape(1, 6), columns=vect.get_feature_names())

,cab,call,me,please,tonight,you
0,1,3,2,1,1,1


In [29]:
# Term frequency–inverse document frequency (simple version)
tf/df

,cab,call,me,please,tonight,you
0,0.0,0.333333,0.0,0.0,1.0,1.0
1,1.0,0.333333,0.5,0.0,0.0,0.0
2,0.0,0.333333,0.5,2.0,0.0,0.0


In [30]:
# TfidfVectorizer
vect = TfidfVectorizer()
pd.DataFrame(vect.fit_transform(simple_train).toarray(), columns=vect.get_feature_names())

,cab,call,me,please,tonight,you
0,0.000000,0.385372,0.000000,0.000000,0.652491,0.652491
1,0.720333,0.425441,0.547832,0.000000,0.000000,0.000000
2,0.000000,0.266075,0.342620,0.901008,0.000000,0.000000


## Pipelines

In [45]:
# Pipeline of transforms with a final estimator. Sequentially apply a list of transforms and a final estimator. 
# Intermediate steps of the pipeline must be ‘transforms’, that is, they must implement fit and transform methods. 
# The final estimator only needs to implement fit.

from sklearn.pipeline import make_pipeline
model = make_pipeline(TfidfVectorizer(lowercase=True,
                                      max_df=1.0, 
                                      min_df=2,
                                      ngram_range=(1,2),
                                      stop_words='english'),
                      LogisticRegression())
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.8776908023483366


/anaconda/lib/python3.6/site-packages/sklearn/linear_model/logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


## Sentiment

In [50]:
tweets = pd.read_csv("./data/Tweets.csv",encoding = "ISO-8859-1")
tweets.head()

,airline_sentiment,airline,text
0,neutral,Virgin America,@VirginAmerica What @dhepburn said.
1,positive,Virgin America,@VirginAmerica plus you've added commercials t...
2,neutral,Virgin America,@VirginAmerica I didn't today... Must mean I n...
3,negative,Virgin America,@VirginAmerica it's really aggressive to blast...
4,negative,Virgin America,@VirginAmerica and it's a really big bad thing...


In [51]:
tweets.airline_sentiment.value_counts()

negative    9178
neutral     3099
positive    2363
Name: airline_sentiment, dtype: int64

In [53]:
# install vaderSentiment first
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

### Sentiment ratings (via https://github.com/cjhutto/vaderSentiment)
Sentiment ratings from 10 independent human raters. Over 9,000 token features were rated on a scale from "[–4] Extremely Negative" to "[4] Extremely Positive", with allowance for "[0] Neutral (or Neither, N/A)". We kept every lexical feature that had a non-zero mean rating, and whose standard deviation was less than 2.5 as determined by the aggregate of those ten independent raters. This left us with just over 7,500 lexical features with validated valence scores that indicated both the sentiment polarity (positive/negative), and the sentiment intensity on a scale from –4 to +4. For example, the word "okay" has a positive valence of 0.9, "good" is 1.9, and "great" is 3.1, whereas "horrible" is –2.5, the frowning emoticon :( is –2.2, and "sucks" and it's slang derivative "sux" are both –1.5.

### Scores
The compound score is computed by summing the valence scores of each word in the lexicon, adjusted according to the rules, and then normalized to be between -1 (most extreme negative) and +1 (most extreme positive). **This is the most useful metric if you want a single unidimensional measure of sentiment for a given sentence. Calling it a 'normalized, weighted composite score' is accurate.**

It is also useful for researchers who would like to set standardized thresholds for classifying sentences as either positive, neutral, or negative. Typical threshold values (used in the literature cited on this page) are:

* positive sentiment: compound score >= 0.05
* neutral sentiment: (compound score > -0.05) and (compound score < 0.05)
* negative sentiment: compound score <= -0.05

**The pos, neu, and neg scores are ratios for proportions of text that fall in each category** (so these should all add up to be 1... or close to it with float operation). These are the most useful metrics if you want multidimensional measures of sentiment for a given sentence.

In [54]:
# get polarity score for example - output is a dictionary
print(tweets['text'][0])
example = sia.polarity_scores(tweets['text'][0])
example

@VirginAmerica What @dhepburn said.


{'compound': 0.0, 'neg': 0.0, 'neu': 1.0, 'pos': 0.0}

In [55]:
# get polarity scores for each observation and add each score to df
def unload_dict(df):
    polarity = sia.polarity_scores(df['text'])
    df['compound'] = polarity['compound']
    df['neg'] = polarity['neg']
    df['neu'] = polarity['neu']
    df['pos'] = polarity['pos']
    return df

tweets = tweets.apply(unload_dict, axis=1)
tweets.sample(3)

,airline_sentiment,airline,text,compound,neg,neu,pos
1481,negative,United,@united airlines: you kidding? Bagage lost thi...,-0.5362,0.24,0.691,0.069
2728,negative,United,@united - #epicfail from a former gate agent i...,0.8510,0.00,0.647,0.353
1372,negative,United,@united Yes I did. We headed out to de-ice 5 ...,0.2144,0.00,0.928,0.072


In [56]:
# this is a very high compound score - note that this is a fallible process
tweets['text'][2728]

'@united - #epicfail from a former gate agent in PIA!  He walked away and quit! Luckily a responsible PIA agent saved the day!'